# Forecasting NSE Stock Prices with a Hybrid EGARCH–XGBoost Model

## 1. Introduction

### 1.1 Background
The Nairobi Securities Exchange (NSE) is Kenya's main equity market. Many NSE stocks trade thinly, have sudden price jumps and go through calm and stormy periods. That makes them hard to forecast with one model alone.

This project combines two models:
* **EGARCH** (Exponential GARCH), an econometric model that captures **volatility**: how much a price is likely to move, and the fact that bad news tends to raise volatility more than good news.
* **XGBoost**, a machine-learning model that learns patterns from **technical features** such as recent returns, momentum, trading volume and price range.

The **hybrid model** feeds EGARCH's volatility estimates into XGBoost as extra inputs.

### 1.2 Objectives
1. Prepare and explore daily NSE price data for the selected stocks.
2. Build standalone EGARCH and XGBoost models and a hybrid EGARCH–XGBoost model.
3. Compare them with simple baselines on data the models have never seen.
4. Deploy the hybrid model as a simple web app where a user picks a stock and gets a forecast.

### 1.3 Data source
* **Source:** daily equity trading data from the Nairobi Securities Exchange. *(Add the exact provider here, for example the NSE data portal or the vendor you used.)*
* **Format:** one Excel file per year, stored in `data/raw/`
* **Period:** January 2017 to June 2026
* **Frequency:** daily (trading days only)
* **Stocks studied:** ABSA Bank Kenya, Britam Holdings, Co-operative Bank, Equity Group, KCB Group, Kenya Airways, KenGen, Kenya Power, plus the remaining selected securities set in `TICKERS` below.

### 1.4 Time split
| Set | Period | Use |
|---|---|---|
| Training | 2017 – 2023 | fit the models |
| Validation | 2024 – 2025 | tune the settings |
| Test | Jan – Jun 2026 | final, untouched evaluation |

The data is split by date, never shuffled, so no model ever sees the future.

In [ ]:
import warnings, re, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
from arch import arch_model          # pip install arch

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 110})
RNG = 42

# ---------------- settings ----------------
DATA_DIR = next((p for p in [Path("data/raw"), Path("../data/raw")] if p.exists()), Path("data/raw"))
FIG_DIR = DATA_DIR.parent.parent / "reports" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

# Your 10 stock codes, exactly as in the CODE column. None = every stock with enough history.
TICKERS = None   # e.g. ["ABSA", "BRIT", "COOP", "EQTY", "KCB", "KQ", "KEGN", "KPLC", "....", "...."]

TRAIN_END = pd.Timestamp("2023-12-31")
VAL_START, VAL_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-12-31")
TEST_START, TEST_END = pd.Timestamp("2026-01-01"), pd.Timestamp("2026-06-30")
EXTREME_RET = 0.15     # daily moves above 15% treated as splits / bonus issues / errors

def savefig(name): plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
print("Data folder:", DATA_DIR.resolve())

## 2. Data variables

### 2.1 Raw variables (as supplied)
| Variable | Type | Description |
|---|---|---|
| `YEAR` | integer | Calendar year of the trading day |
| `DATE` | date | Trading date |
| `CODE` | text | Stock ticker, e.g. SCOM, KQ |
| `NAME` | text | Company name |
| `12m Low` | KES | Lowest price over the past 12 months |
| `12m High` | KES | Highest price over the past 12 months |
| `Day Low` | KES | Lowest traded price that day |
| `Day High` | KES | Highest traded price that day |
| `Day Price` | KES | Closing price that day (**main variable**) |
| `Previous` | KES | Previous day's closing price |
| `Change` | KES | Day Price minus Previous |
| `Change%` | % | Percentage change from Previous |
| `Volume` | shares | Number of shares traded that day |

### 2.2 Derived variables (created in this notebook)
| Variable | Formula / meaning |
|---|---|
| `ret` | Daily log return, $r_t = \ln(P_t / P_{t-1})$ |
| `target` | Next day's log return $r_{t+1}$ (what the models predict) |
| `ret_lag1..5` | Returns over the last 5 days |
| `ret_ma5/21`, `mom_63` | Short-term average return and 3-month momentum |
| `vol_5/21/63` | Rolling standard deviation of returns (realised volatility) |
| `range_pct` | (Day High − Day Low) / Day Price, the size of the day's trading range |
| `px_ma20_gap` | How far the price sits above or below its 20-day average |
| `pos_12m` | Where the price sits in its 12-month range (0 = at the low, 1 = at the high) |
| `rsi_14` | 14-day Relative Strength Index |
| `logvol_z` | Today's volume compared with its usual level (z-score) |
| `stale_5` | Days with no price change in the last week (a thin-trading signal) |
| `mkt_ret` | Average return of all selected stocks that day (market factor) |
| `eg_vol`, `eg_resid` | EGARCH volatility forecast and standardised shock (hybrid inputs) |

## 3. Data preparation
### 3.1 Load all yearly files

In [ ]:
COLMAP = {"year": "year", "date": "date", "code": "code", "name": "name", "12mlow": "low_12m",
          "12mhigh": "high_12m", "daylow": "day_low", "dayhigh": "day_high", "dayprice": "price",
          "previous": "previous", "change": "change", "change%": "change_pct", "volume": "volume"}

def to_num(s):
    if s.dtype.kind in "if": return s.astype(float)
    return pd.to_numeric(s.astype(str).str.replace(",", "").str.replace("%", "").str.strip()
                         .replace({"-": None, "": None, "nan": None}), errors="coerce")

files = sorted(f for f in glob.glob(str(DATA_DIR / "*"))
               if f.lower().endswith((".xlsx", ".xls", ".csv")) and not Path(f).name.startswith("~$"))
assert files, f"No data files in {DATA_DIR.resolve()}"

frames = []
for f in files:
    d = pd.read_csv(f) if f.lower().endswith(".csv") else pd.read_excel(f)
    d.columns = [COLMAP.get(re.sub(r"[^a-z0-9%]", "", str(c).lower()), str(c)) for c in d.columns]
    frames.append(d); print(f"{Path(f).name:30s} {len(d):>8,} rows")

raw = pd.concat(frames, ignore_index=True)
raw["date"] = pd.to_datetime(raw["date"], dayfirst=True, errors="coerce")
raw["code"] = raw["code"].astype(str).str.strip().str.upper()
raw["name"] = raw["name"].astype(str).str.strip()
for c in ["low_12m", "high_12m", "day_low", "day_high", "price", "previous", "change", "change_pct", "volume"]:
    if c in raw: raw[c] = to_num(raw[c])
print("\nCombined:", raw.shape)
raw.head()

### 3.2 Data quality checks

In [ ]:
quality = pd.DataFrame({"missing": raw.isna().sum(), "missing_%": raw.isna().mean() * 100})
display(quality)
print("Unreadable dates       :", raw["date"].isna().sum())
print("Duplicate (date, code) :", raw.duplicated(["date", "code"]).sum())
print("Zero / negative prices :", (raw["price"] <= 0).sum())
print("Price outside day range:", ((raw["price"] < raw["day_low"] - 1e-9) | (raw["price"] > raw["day_high"] + 1e-9)).sum())
print("Date range             :", raw["date"].min().date(), "to", raw["date"].max().date())

### 3.3 Cleaning, stock selection and returns

In [ ]:
df = (raw.dropna(subset=["date", "price"]).query("price > 0")
         .sort_values(["code", "date"]).drop_duplicates(["date", "code"], keep="last"))

if TICKERS:
    missing = set(TICKERS) - set(df["code"])
    if missing: print("Not found in data:", missing)
    df = df[df["code"].isin(TICKERS)]
else:
    n = df.groupby("code")["date"].nunique()
    df = df[df["code"].isin(n[n >= 1000].index)]
df = df.reset_index(drop=True)

df["ret"] = df.groupby("code")["price"].transform(lambda p: np.log(p).diff())
df["extreme"] = df["ret"].abs() > EXTREME_RET
df["year"] = df["date"].dt.year
df["turnover"] = df["price"] * df["volume"].fillna(0)

NAMES = df.groupby("code")["name"].last().to_dict()
TICKS = sorted(df["code"].unique())
print(f"Stocks: {len(TICKS)} -> {TICKS}")
print(f"Rows: {len(df):,} | extreme moves flagged: {df['extreme'].sum()}")
display(df.loc[df["extreme"], ["date", "code", "previous", "price", "ret"]].head(15))

## 4. Exploratory data analysis
### 4.1 Summary by stock

In [ ]:
summary = (df.groupby("code").agg(
    name=("name", "last"), first_date=("date", "min"), last_date=("date", "max"),
    trading_days=("date", "nunique"), min_price=("price", "min"), max_price=("price", "max"),
    last_price=("price", "last"), median_volume=("volume", "median"),
    zero_change_days=("ret", lambda r: (r == 0).mean()),
    annual_return=("ret", lambda r: r[r.abs() <= EXTREME_RET].mean() * 252),
    annual_volatility=("ret", lambda r: r[r.abs() <= EXTREME_RET].std() * np.sqrt(252))))
summary.sort_values("median_volume", ascending=False)

### 4.2 Average price per stock per year
The main view of how each stock's price level has changed between 2017 and 2026.

In [ ]:
avg_year = df.pivot_table(index="year", columns="code", values="price", aggfunc="mean")
display(avg_year.round(2))

# (a) grouped bar chart
ax = avg_year.T.plot.bar(figsize=(16, 6), width=0.85, colormap="viridis")
ax.set_title("Average closing price per stock per year (KES)"); ax.set_xlabel(""); ax.set_ylabel("KES")
ax.legend(title="Year", ncol=5, fontsize=8); plt.xticks(rotation=0)
plt.tight_layout(); savefig("01_avg_price_by_year_bar"); plt.show()

In [ ]:
# (b) one small panel per stock: each has its own price scale, so cheap stocks are readable too
cols = 5; rows = int(np.ceil(len(TICKS) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(18, 3.4 * rows), sharex=True)
for ax, t in zip(axes.flat, TICKS):
    s = avg_year[t].dropna()
    ax.bar(s.index, s.values, color="#2a9d8f"); ax.plot(s.index, s.values, "o-", color="#264653", lw=1)
    ax.set_title(f"{t}", fontsize=10, weight="bold"); ax.tick_params(axis="x", rotation=45, labelsize=8)
for ax in list(axes.flat)[len(TICKS):]: ax.set_visible(False)
fig.suptitle("Average yearly price by stock (KES, own scale)", y=1.01, fontsize=13)
plt.tight_layout(); savefig("02_avg_price_by_year_panels"); plt.show()

In [ ]:
# (c) heatmap of year-on-year % change in average price
yoy = avg_year.pct_change() * 100
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(yoy.T, annot=True, fmt=".0f", cmap="RdYlGn", center=0, ax=ax, cbar_kws={"label": "% change"})
ax.set_title("Year-on-year change in average price (%)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); savefig("03_yoy_change_heatmap"); plt.show()

### 4.3 Price trends over time

In [ ]:
piv_px = df.pivot(index="date", columns="code", values="price").sort_index()
rebased = piv_px.apply(lambda s: s / s.dropna().iloc[0] * 100)
fig, ax = plt.subplots(figsize=(15, 6))
rebased.plot(ax=ax, lw=1.1, logy=True)
ax.axhline(100, color="k", lw=0.6, ls="--")
for d, lab in [(VAL_START, "Validation"), (TEST_START, "Test")]:
    ax.axvline(d, color="grey", ls=":"); ax.text(d, ax.get_ylim()[1] * 0.8, f" {lab}", color="grey")
ax.set_title("Prices rebased to 100 at start (log scale)"); ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); savefig("04_rebased_prices"); plt.show()

### 4.4 Trading activity and liquidity

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
summary["median_volume"].sort_values().plot.barh(ax=ax[0], color="#264653", logx=True)
ax[0].set_title("Median daily volume (shares, log scale)")
summary["zero_change_days"].sort_values().plot.barh(ax=ax[1], color="#f4a261")
ax[1].set_title("Share of days with no price change"); ax[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
vol_year = df.pivot_table(index="year", columns="code", values="turnover", aggfunc="sum") / 1e9
vol_year.sum(axis=1).plot.bar(ax=ax[2], color="#2a9d8f"); ax[2].set_title("Total turnover of selected stocks (KES bn)")
plt.tight_layout(); savefig("05_liquidity"); plt.show()

### 4.5 Return distributions

In [ ]:
rc = df[~df["extreme"]].dropna(subset=["ret"])
dist = rc.groupby("code")["ret"].agg(
    mean_pct=lambda r: r.mean() * 100, std_pct=lambda r: r.std() * 100,
    skew=stats.skew, excess_kurtosis=stats.kurtosis,
    jarque_bera_p=lambda r: stats.jarque_bera(r).pvalue)
display(dist)

fig, axes = plt.subplots(2, 5, figsize=(20, 7)); 
for ax, t in zip(axes.flat, TICKS[:10]):
    r = rc.loc[rc["code"] == t, "ret"]
    ax.hist(r, bins=100, density=True, color="#264653", alpha=0.6)
    x = np.linspace(r.quantile(0.001), r.quantile(0.999), 200)
    ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), color="#e76f51", label="Normal")
    ax.set_xlim(x.min(), x.max()); ax.set_title(t)
for ax in list(axes.flat)[len(TICKS):]: ax.set_visible(False)
axes.flat[0].legend(); fig.suptitle("Daily returns vs the normal curve: tall peaks and fat tails", y=1.01)
plt.tight_layout(); savefig("06_return_distributions"); plt.show()
print("JB p < 0.05 means returns are not normal, which is why EGARCH uses Student-t errors.")

### 4.6 Volatility clustering and ARCH effects

In [ ]:
piv_ret = rc.pivot(index="date", columns="code", values="ret").sort_index()
roll_vol = piv_ret.rolling(21, min_periods=15).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(15, 5))
sns.heatmap(roll_vol.resample("M").mean().T, cmap="magma_r", ax=ax, cbar_kws={"label": "annualised volatility"})
ax.set_xticklabels([t.get_text()[:4] for t in ax.get_xticklabels()])
ax.set_title("Monthly volatility by stock: calm and stormy periods cluster together"); ax.set_xlabel("")
plt.tight_layout(); savefig("07_volatility_heatmap"); plt.show()

t0 = summary["median_volume"].idxmax(); r0 = piv_ret[t0].dropna()
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(r0, lags=30, ax=ax[0], title=f"{t0}: autocorrelation of returns")
plot_acf(r0 ** 2, lags=30, ax=ax[1], title=f"{t0}: autocorrelation of squared returns")
plt.tight_layout(); savefig("08_acf"); plt.show()

tests = pd.DataFrame({t: {
    "ADF_price_p": adfuller(np.log(piv_px[t].dropna()))[1],
    "ADF_return_p": adfuller(piv_ret[t].dropna())[1],
    "LjungBox_sq_p": acorr_ljungbox(piv_ret[t].dropna() ** 2, lags=[10]).iloc[0, 1],
    "ARCH_LM_p": het_arch(piv_ret[t].dropna())[1]} for t in TICKS}).T
display(tests)
print("Prices are non-stationary while returns are stationary, so we model returns.")
print(f"ARCH effects significant (p < 0.05) in {(tests['ARCH_LM_p'] < 0.05).sum()}/{len(tests)} stocks. "
      "Where significant, volatility is predictable, which supports a GARCH-type model.")

### 4.7 Correlation between stocks and calendar patterns

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(piv_ret.loc[:TRAIN_END].corr(min_periods=250), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[0])
ax[0].set_title("Correlation of daily returns (training period)")
cal = rc.assign(weekday=rc["date"].dt.day_name().str[:3])
sns.barplot(data=cal, x="weekday", y="ret", order=["Mon", "Tue", "Wed", "Thu", "Fri"], ax=ax[1], color="#2a9d8f")
ax[1].set_title("Average return by weekday (95% CI)")
plt.tight_layout(); savefig("09_correlation_weekday"); plt.show()

### 4.8 EDA takeaways
* Price levels and trends differ a lot between stocks, so every model works with **returns**, not prices.
* Returns have **fat tails**, so EGARCH uses Student-t errors.
* Volatility **clusters**: there are clear ARCH effects, which supports EGARCH.
* Many stocks have frequent no-change days (thin trading). That creates short-term patterns XGBoost may pick up.

## 5. Modelling
### 5.1 Feature engineering
Each feature for day *t* uses information up to day *t* only. The target is day *t+1*'s return.

In [ ]:
def rsi(p, n=14):
    d = p.diff(); up = d.clip(lower=0).ewm(alpha=1/n).mean(); dn = (-d.clip(upper=0)).ewm(alpha=1/n).mean()
    return 100 - 100 / (1 + up / dn.replace(0, np.nan))

feat = df.merge(rc.groupby("date")["ret"].mean().rename("mkt_ret"), on="date", how="left")
feat = feat.sort_values(["code", "date"]).reset_index(drop=True)
feat.loc[feat["extreme"], "ret"] = np.nan
g = feat.groupby("code")
roll = lambda col, w, fn, mp=None: g[col].transform(lambda s: getattr(s.rolling(w, min_periods=mp or w // 2), fn)())

for k in range(1, 6): feat[f"ret_lag{k}"] = g["ret"].shift(k - 1)
feat["ret_ma5"], feat["ret_ma21"] = roll("ret", 5, "mean"), roll("ret", 21, "mean")
feat["mom_63"] = roll("ret", 63, "sum", 40)
for w in [5, 21, 63]: feat[f"vol_{w}"] = roll("ret", w, "std")
feat["range_pct"] = (feat["day_high"] - feat["day_low"]) / feat["price"]
feat["px_ma20_gap"] = feat["price"] / roll("price", 20, "mean", 20) - 1
feat["pos_12m"] = (feat["price"] - feat["low_12m"]) / (feat["high_12m"] - feat["low_12m"]).replace(0, np.nan)
feat["rsi_14"] = g["price"].transform(rsi)
feat["lv"] = np.log1p(feat["volume"].fillna(0))
feat["logvol_z"] = (feat["lv"] - roll("lv", 63, "mean", 20)) / roll("lv", 63, "std", 20)
feat["is_stale"] = (feat["ret"] == 0).astype(float); feat["stale_5"] = roll("is_stale", 5, "sum", 5)
feat["dow"] = feat["date"].dt.dayofweek
feat["target"] = g["ret"].shift(-1)
feat["code_cat"] = pd.Categorical(feat["code"], categories=TICKS)
feat = feat.replace([np.inf, -np.inf], np.nan)

BASE = [f"ret_lag{k}" for k in range(1, 6)] + ["ret_ma5", "ret_ma21", "mom_63", "vol_5", "vol_21", "vol_63",
        "range_pct", "px_ma20_gap", "pos_12m", "rsi_14", "logvol_z", "stale_5", "mkt_ret", "dow", "code_cat"]
print(len(BASE), "features"); feat[BASE[:-1] + ["target"]].describe().T

### 5.2 Train / validation / test split

In [ ]:
data = feat.dropna(subset=["target"]).copy()
data["split"] = np.select([data["date"] <= TRAIN_END, data["date"].between(VAL_START, VAL_END),
                           data["date"].between(TEST_START, TEST_END)], ["train", "val", "test"], "out")
data = data[data["split"] != "out"].reset_index(drop=True)
data.groupby("split").agg(rows=("target", "size"), start=("date", "min"), end=("date", "max")).loc[["train", "val", "test"]]

### 5.3 Evaluation metrics and baselines
* **RMSE / MAE** of the next-day return (%)
* **Directional accuracy**: how often the predicted up/down direction is right (days with a price change only)
* **Price MAPE**: error in the implied price $\hat P_{t+1} = P_t e^{\hat r_{t+1}}$

Baselines: **random walk** (predicts no change) and **historical mean** (each stock's average training return).

In [ ]:
def evaluate(d, col, name):
    m = d.dropna(subset=[col, "target"]); y, p = m["target"], m[col]; nz = y != 0
    return {"model": name,
            "RMSE_%": np.sqrt(mean_squared_error(y, p)) * 100, "MAE_%": mean_absolute_error(y, p) * 100,
            "Direction_acc": (np.sign(p[nz]) == np.sign(y[nz])).mean() if (p != 0).any() else np.nan,
            "Price_MAPE_%": (abs(np.exp(p) - np.exp(y)) / np.exp(y)).mean() * 100}

data["pred_rw"] = 0.0
data["pred_mean"] = data["code"].map(data[data["split"] == "train"].groupby("code")["target"].mean())
results = []
def log_results(col, name):
    for s in ["val", "test"]: results.append({"split": s, **evaluate(data[data["split"] == s], col, name)})
log_results("pred_rw", "Random walk"); log_results("pred_mean", "Historical mean")
pd.DataFrame(results)

### 5.4 EGARCH(1,1,1) with Student-t errors
One model per stock, fitted on **training data only**:

$$r_t = \mu + \phi r_{t-1} + \varepsilon_t,\qquad \varepsilon_t = \sigma_t z_t,\ z_t\sim t_\nu$$
$$\ln\sigma_t^2 = \omega + \alpha\big(|z_{t-1}| - E|z|\big) + \gamma z_{t-1} + \beta\ln\sigma_{t-1}^2$$

* $\beta$ measures **persistence**: how long volatility shocks last.
* $\gamma<0$ is the **leverage effect**: bad news raises volatility more than good news.

After training the parameters are frozen, and each day's forecast uses only past data.

In [ ]:
eg_params, eg_out = {}, []
for t in TICKS:
    s = (feat.loc[feat["code"] == t].set_index("date")["ret"].dropna() * 100).clip(-10, 10)
    s = s[s.index <= TEST_END]
    if (s.index <= TRAIN_END).sum() < 500: print("skip", t); continue
    res = arch_model(s, mean="AR", lags=1, vol="EGARCH", p=1, o=1, q=1, dist="t", rescale=False) \
            .fit(last_obs=s.index[s.index > TRAIN_END][0], disp="off", options={"maxiter": 500})
    eg_params[t] = res.params
    fc = res.forecast(horizon=1, start=s.index[1], reindex=False)
    out = pd.DataFrame({"date": fc.mean.index, "code": t,
                        "eg_mean": fc.mean.iloc[:, 0].values / 100,
                        "eg_vol": np.sqrt(fc.variance.iloc[:, 0].values) / 100})
    p = res.params; resid = s - p.iloc[0] - p.iloc[1] * s.shift(1)
    prev_sigma = pd.Series(out["eg_vol"].values * 100, index=out["date"]).shift(1)
    out["eg_resid"] = (resid.reindex(out["date"]) / prev_sigma).values
    eg_out.append(out)

eg_params = pd.DataFrame(eg_params).T
data = data.merge(pd.concat(eg_out), on=["date", "code"], how="left")
display(eg_params.round(4))

In [ ]:
beta = eg_params.filter(like="beta").iloc[:, 0]; gamma = eg_params.filter(like="gamma").iloc[:, 0]
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
beta.sort_values().plot.barh(ax=ax[0], color="#264653"); ax[0].set_title("β: volatility persistence")
gamma.sort_values().plot.barh(ax=ax[1], color=np.where(gamma.sort_values() < 0, "#e76f51", "#2a9d8f"))
ax[1].axvline(0, color="k", lw=0.7); ax[1].set_title("γ: asymmetry (negative = leverage effect)")
eg_params["nu"].sort_values().plot.barh(ax=ax[2], color="#e9c46a"); ax[2].set_title("ν: tail thickness (lower = fatter)")
plt.tight_layout(); savefig("10_egarch_params"); plt.show()

d = data[(data["code"] == t0) & (data["split"] != "train")]
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(d["date"], d["target"].abs() * 100, color="grey", lw=0.6, label="|actual return| %")
ax.plot(d["date"], d["eg_vol"] * 100, color="#e76f51", lw=1.5, label="EGARCH volatility forecast %")
ax.axvline(TEST_START, color="k", ls=":"); ax.legend(); ax.set_title(f"{t0}: out-of-sample volatility forecast")
plt.tight_layout(); savefig("11_egarch_volatility"); plt.show()

data["pred_egarch"] = data["eg_mean"]; log_results("pred_egarch", "EGARCH")

### 5.5 XGBoost (standalone) and 5.6 Hybrid EGARCH–XGBoost
* **XGBoost** is one pooled model across all stocks, with the stock code as a category. Pooling gives it far more data than separate per-stock models would.
* The **hybrid** uses the same features **plus** `eg_vol`, `eg_mean`, `eg_resid` and `vol_gap` (= `eg_vol` / `vol_21`).
* **Tuning:** a small grid, scored on validation RMSE, with early stopping.
* **Final model:** refit on train + validation, then scored once on test.

In [ ]:
from itertools import product
grid_log = []

def fit_xgb(features, label):
    tr, va = data[data["split"] == "train"], data[data["split"] == "val"]
    common = dict(subsample=0.8, colsample_bytree=0.8, objective="reg:pseudohubererror", huber_slope=0.02,
                  tree_method="hist", enable_categorical=True, random_state=RNG, n_jobs=-1)
    best = None
    for depth, lr, mcw in product([3, 5], [0.03, 0.1], [5, 50]):
        m = xgb.XGBRegressor(n_estimators=1500, max_depth=depth, learning_rate=lr, min_child_weight=mcw,
                             early_stopping_rounds=100, **common)
        m.fit(tr[features], tr["target"], eval_set=[(va[features], va["target"])], verbose=False)
        rmse = np.sqrt(mean_squared_error(va["target"], m.predict(va[features]))) * 100
        grid_log.append({"model": label, "max_depth": depth, "lr": lr, "min_child_weight": mcw,
                         "trees": m.best_iteration + 1, "val_RMSE_%": rmse})
        if best is None or rmse < best[0]: best = (rmse, m, depth, lr, mcw)
    _, tuned, depth, lr, mcw = best
    final = xgb.XGBRegressor(n_estimators=max(tuned.best_iteration + 1, 50), max_depth=depth,
                             learning_rate=lr, min_child_weight=mcw, **common)
    trva = data[data["split"].isin(["train", "val"])]; final.fit(trva[features], trva["target"])
    pred = pd.Series(tuned.predict(data[features]), index=data.index)
    pred[data["split"] == "test"] = final.predict(data.loc[data["split"] == "test", features])
    print(f"{label}: depth={depth}, lr={lr}, min_child_weight={mcw}, trees={tuned.best_iteration + 1}")
    return tuned, pred

xgb_model, data["pred_xgb"] = fit_xgb(BASE, "XGBoost"); log_results("pred_xgb", "XGBoost")

data["vol_gap"] = data["eg_vol"] / data["vol_21"]
HYBRID = BASE + ["eg_vol", "eg_mean", "eg_resid", "vol_gap"]
hyb_model, data["pred_hybrid"] = fit_xgb(HYBRID, "Hybrid"); log_results("pred_hybrid", "Hybrid EGARCH-XGBoost")
pd.DataFrame(grid_log).sort_values(["model", "val_RMSE_%"]).groupby("model").head(3)

### 5.7 Model comparison

In [ ]:
res_tbl = pd.DataFrame(results).drop_duplicates(["split", "model"], keep="last").set_index(["split", "model"])
display(res_tbl.style.format({"RMSE_%": "{:.4f}", "MAE_%": "{:.4f}", "Direction_acc": "{:.1%}", "Price_MAPE_%": "{:.3f}"})
        .highlight_min(subset=["RMSE_%", "MAE_%", "Price_MAPE_%"], color="#d8f3dc")
        .highlight_max(subset=["Direction_acc"], color="#d8f3dc"))

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5)); t = res_tbl.reset_index()
for a, m in zip(ax, ["RMSE_%", "MAE_%", "Direction_acc"]):
    sns.barplot(data=t, x="model", y=m, hue="split", ax=a, palette=["#e9c46a", "#e76f51"])
    lo, hi = t[m].min(), t[m].max(); a.set_ylim(lo - (hi - lo) * 0.6, hi + (hi - lo) * 0.3)
    a.set_title(m); a.set_xlabel(""); a.tick_params(axis="x", rotation=25)
ax[2].axhline(0.5, color="k", ls="--", lw=0.8)
plt.tight_layout(); savefig("12_model_comparison"); plt.show()

In [ ]:
# Diebold-Mariano test: is the difference in squared errors statistically significant?
def dm_test(y, a, b):
    d = ((y - a) ** 2 - (y - b) ** 2).dropna(); T = len(d)
    stat = d.mean() / np.sqrt(d.var(ddof=0) / T) * np.sqrt((T - 1) / T)
    return stat, 2 * stats.t.sf(abs(stat), T - 1)

P = {"Random walk": "pred_rw", "EGARCH": "pred_egarch", "XGBoost": "pred_xgb", "Hybrid": "pred_hybrid"}
rows = []
te = data[data["split"] == "test"].dropna(subset=list(P.values()))
for a, b in [("Hybrid", "Random walk"), ("Hybrid", "XGBoost"), ("Hybrid", "EGARCH"), ("XGBoost", "Random walk")]:
    s, p = dm_test(te["target"], te[P[a]], te[P[b]])
    rows.append({"comparison": f"{a} vs {b}", "DM_stat": s, "p_value": p,
                 "result": (f"{a} better" if s < 0 else f"{b} better") if p < 0.05 else "no significant difference"})
pd.DataFrame(rows)

In [ ]:
# Per-stock test RMSE and feature importance
per = pd.DataFrame({k: te.groupby("code").apply(lambda d: np.sqrt(mean_squared_error(d["target"], d[c])) * 100)
                    for k, c in P.items()})
fig, ax = plt.subplots(1, 2, figsize=(18, 6))
per.plot.bar(ax=ax[0], colormap="viridis"); ax[0].set_title("Test RMSE (%) by stock"); ax[0].set_xlabel("")
imp = pd.Series(hyb_model.get_booster().get_score(importance_type="gain")).reindex(HYBRID).fillna(0).sort_values()
imp.plot.barh(ax=ax[1], color=["#e76f51" if c.startswith("eg_") or c == "vol_gap" else "#264653" for c in imp.index])
ax[1].set_title("Hybrid feature importance (EGARCH inputs in orange)")
plt.tight_layout(); savefig("13_per_stock_importance"); plt.show()

In [ ]:
# Actual vs forecast price in the test period
d = data[(data["code"] == t0) & (data["split"] == "test")]
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(d["date"], d["price"] * np.exp(d["target"]), color="k", lw=1.5, label="Actual price")
ax.plot(d["date"], d["price"] * np.exp(d["pred_hybrid"]), color="#e76f51", ls=":", lw=2, label="Hybrid forecast")
ax.fill_between(d["date"], d["price"] * np.exp(-1.28 * d["eg_vol"]), d["price"] * np.exp(1.28 * d["eg_vol"]),
                color="#e76f51", alpha=0.15, label="80% EGARCH range")
ax.set_title(f"{t0}: next-day forecasts, Jan–Jun 2026"); ax.legend()
plt.tight_layout(); savefig("14_test_forecast"); plt.show()

## 6. Deployment

### 6.1 How it's deployed
The hybrid model is served through a **Streamlit web app** (`app.py`) packaged in **Docker** (`Dockerfile.app`), so it runs the same way on any computer.

```
Excel files (data/raw)  →  app.py  →  EGARCH volatility + Hybrid XGBoost  →  web page (port 8501)
                             ▲
                         Docker container (python:3.12-slim)
```

**Kept simple for users:** two drop-downs (stock and forecast period), one **Forecast** button, three headline numbers, a chart with the forecast as a dotted line, and a disclaimer.

### 6.2 From a 1-day model to multi-day forecasts
The model is evaluated **one day ahead**. The app extends this to 1 day, 1 week, 2 weeks, 1 month and 3 months:
1. **Day 1:** return from the hybrid model, capped at ±2× today's volatility.
2. **Later days:** the day-1 signal halves each day and moves toward the stock's small long-run trend. Daily returns carry very little signal beyond a few days.
3. **Likely range (80%):** the daily volatility starts at EGARCH's forecast and fades back to the stock's normal level over the last 12 months, kept within ¼× to 3× of that level. The range widens with the square root of the horizon.

These safeguards came from testing. An earlier version used multi-step EGARCH simulation, which produced unrealistic ranges on thinly traded, jumpy stocks such as Kenya Airways.

The cell below reproduces the app's forecast logic so it can be shown here.

In [ ]:
HORIZONS = {"1 day": 1, "1 week": 5, "2 weeks": 10, "1 month": 21, "3 months": 63}

def app_forecast(code, horizon, z=1.28):
    s = df[df["code"] == code].set_index("date")
    r = feat.loc[feat["code"] == code].set_index("date")["ret"].dropna()
    res = arch_model((r * 100).clip(-10, 10), mean="Constant", vol="EGARCH", p=1, o=1, q=1, dist="t",
                     rescale=False).fit(disp="off")
    vol_long = r.tail(252).std()
    vol_now = np.clip(np.sqrt(res.forecast(horizon=1, reindex=False).variance.iloc[-1, 0]) / 100,
                      0.25 * vol_long, 3 * vol_long)
    b = np.clip(res.params.filter(like="beta").iloc[0], 0.5, 0.98); k = np.arange(horizon)
    daily_vol = np.sqrt(vol_long ** 2 + (vol_now ** 2 - vol_long ** 2) * b ** k)
    last = data[data["code"] == code].dropna(subset=["pred_hybrid"]).iloc[-1]
    next_ret = np.clip(last["pred_hybrid"], -2 * vol_now, 2 * vol_now)
    drift = np.clip(r.tail(252).mean(), -0.002, 0.002)
    cum = np.cumsum(drift + (next_ret - drift) * 0.5 ** k); sd = np.sqrt(np.cumsum(daily_vol ** 2))
    px = s["price"].iloc[-1]
    return pd.DataFrame({"date": pd.bdate_range(s.index[-1] + pd.Timedelta(days=1), periods=horizon),
                         "forecast": px * np.exp(cum), "low": px * np.exp(cum - z * sd),
                         "high": px * np.exp(cum + z * sd)}), s["price"], px

rows = []
for t in TICKS:
    for lab, h in HORIZONS.items():
        p, _, px = app_forecast(t, h)
        rows.append({"stock": t, "horizon": lab, "last_price": px, "forecast": p["forecast"].iloc[-1],
                     "low": p["low"].iloc[-1], "high": p["high"].iloc[-1]})
fc_tbl = pd.DataFrame(rows); fc_tbl["range_width_%"] = (fc_tbl["high"] / fc_tbl["low"] - 1) * 100
fc_tbl.pivot(index="stock", columns="horizon", values="range_width_%")[list(HORIZONS)].round(1)

In [ ]:
# What the app chart looks like: one stock, 1 month and 3 months
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, lab in zip(axes, ["1 month", "3 months"]):
    p, hist, px = app_forecast(t0, HORIZONS[lab]); hist = hist.tail(90)
    x = [hist.index[-1]] + list(p["date"])
    ax.plot(hist.index, hist.values, color="#264653", lw=2, label="Actual price")
    ax.plot(x, [px] + list(p["forecast"]), color="#e76f51", ls=":", lw=2.5, label="Forecast")
    ax.fill_between(x, [px] + list(p["low"]), [px] + list(p["high"]), color="#e76f51", alpha=0.15, label="Likely range (80%)")
    ax.set_title(f"{NAMES[t0]}: {lab} forecast"); ax.legend(); ax.set_ylabel("KES")
plt.tight_layout(); savefig("15_app_forecast_preview"); plt.show()

### 6.3 Running the deployed app
```bash
# On your machine
pip install -r requirements-app.txt
streamlit run app.py                     # opens http://localhost:8501

# In Docker
docker build -f Dockerfile.app -t nse-app .
docker run --rm -p 8501:8501 nse-app     # open http://localhost:8501
```

### 6.4 Limitations and next steps
* EGARCH parameters are fixed after training. Re-estimating them monthly could help.
* Longer-horizon forecasts lean mostly on EGARCH volatility and the long-run trend. The hybrid model's advantage is mainly at 1 day.
* The app re-trains when a stock is first selected. In production, models could be pre-trained daily and stored.
* Possible extensions: automatic daily data refresh, quantile XGBoost for prediction ranges, and SHAP explanations.

### 6.5 Disclaimer
These forecasts are for academic research only and are **not financial advice**. Stock prices are uncertain, and actual prices can move outside the ranges shown.